In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
TC= pd.read_csv('../data/raw/ForExport.csv')
TC

In [ ]:
TC.shape

In [ ]:
TC['City'].unique()

In [ ]:
uae_cities = [
    'abu-dhabi',
    'dubai',
    'sharjah',
    'fujairah',
    'ras-al-khaimah',
    'al-ain'
]

uae_df = TC[TC['City'].isin(uae_cities)]

In [ ]:
print(uae_df['City'].unique())

In [ ]:
uae_df.shape

In [ ]:
uae_df = uae_df.drop(columns=['Country'])

In [ ]:
uae_df.columns

In [ ]:
uae_df.isnull().sum()

In [ ]:
uae_df['UpdateTimeUTC'] = pd.to_datetime(uae_df['UpdateTimeUTC'])

In [ ]:
uae_df['Hour'] = uae_df['UpdateTimeUTC'].dt.hour
uae_df['DayName'] = uae_df['UpdateTimeUTC'].dt.day_name()
uae_df['Month'] = uae_df['UpdateTimeUTC'].dt.month_name()

In [ ]:
uae_df.info()

In [ ]:
uae_df = uae_df.rename(columns={
    'UpdateTimeUTC': 'Timestamp',
    'JamsDelay': 'TotalJamDelay',
    'TrafficIndexLive': 'TrafficIndex',
    'JamsLengthInKms': 'JamLengthKm',
    'JamsCount': 'JamCount',
    'TrafficIndexWeekAgo': 'TrafficIndexLastWeek',
    'TravelTimeLivePer10KmsMins': 'LiveTravelTimePer10Km',
    'TravelTimeHistoricPer10KmsMins': 'HistoricTravelTimePer10Km',
    'MinsDelay': 'TravelDelayMinutes'
})

In [ ]:
uae_df.to_csv('uae_traffic_cleaned.csv', index=False)

In [ ]:
uae_df.head()

In [ ]:
uae_df.describe()

In [ ]:
# Group by DayOfWeek and Hour to see if the spikes repeat consistently
traffic_variance = uae_df.groupby(['DayName', 'Hour'])['TravelDelayMinutes'].agg(['median', 'mean', 'std', 'max'])
print(traffic_variance.head(24))

In [ ]:
uae_df['TrafficFrictionScore'] = (
    (
        uae_df['LiveTravelTimePer10Km']
        - uae_df['HistoricTravelTimePer10Km']
    )
    / uae_df['HistoricTravelTimePer10Km']
) * uae_df['TrafficIndex']

In [ ]:
uae_df['TrafficFrictionScore'].describe()

In [ ]:
city_friction = (
    uae_df.groupby('City')['TrafficFrictionScore']
    .mean()
    .sort_values(ascending=False)
)

print(city_friction)

In [ ]:
hourly_friction = (
    uae_df.groupby('Hour')['TrafficFrictionScore']
    .mean()
)

print(hourly_friction)

In [ ]:
def friction_category(score):
    if score < 0:
        return 'Low'
    elif score < 2:
        return 'Moderate'
    elif score < 5:
        return 'High'
    else:
        return 'Severe'

uae_df['FrictionCategory'] = (
    uae_df['TrafficFrictionScore']
    .apply(friction_category)
)

In [ ]:
uae_df['FrictionCategory'].value_counts()

In [ ]:
uae_df.to_csv('uae_traffic_kpi_engineered.csv', index=False)

In [ ]:
df =pd.read_csv('../data/processed/uae_traffic_kpi_engineered.csv')

In [ ]:
df.head()

In [ ]:


df['TravelTimeInflationPct'] = (
    (
        df['LiveTravelTimePer10Km']
        -
        df['HistoricTravelTimePer10Km']
    )
    /
    df['HistoricTravelTimePer10Km']
) * 100

# Round values for readability
df['TravelTimeInflationPct'] = (
    df['TravelTimeInflationPct']
    .round(2)
)


print(
    df['TravelTimeInflationPct']
    .describe()
)

In [ ]:
df.groupby('City')['TravelTimeInflationPct'] \
.mean() \
.sort_values(ascending=False)

In [ ]:

volatility_idx = (
    df.groupby(['City', 'Hour'])['TrafficFrictionScore']
    .std()
    .reset_index()
)


volatility_idx.rename(
    columns={
        'TrafficFrictionScore': 'FrictionVolatilityIndex'
    },
    inplace=True
)

volatility_idx['FrictionVolatilityIndex'] = (
    volatility_idx['FrictionVolatilityIndex']
    .fillna(0)
)


df = df.merge(
    volatility_idx,
    on=['City', 'Hour'],
    how='left'
)

print(
    df['FrictionVolatilityIndex']
    .describe()
)

In [ ]:
df.groupby('City')['FrictionVolatilityIndex'] \
.mean() \
.sort_values(ascending=False)

In [ ]:
recovery_matrix = (
    df.groupby('City')['TrafficFrictionScore']
    .agg(['median', 'max'])
    .reset_index()
)

# Calculate escalation severity
recovery_matrix['NetworkShockEscalation'] = (
    recovery_matrix['max']
    -
    recovery_matrix['median']
).round(2)

# Merge into dataframe
df = df.merge(
    recovery_matrix[['City', 'NetworkShockEscalation']],
    on='City',
    how='left'
)

print(
    recovery_matrix[
        ['City', 'NetworkShockEscalation']
    ].sort_values(
        by='NetworkShockEscalation',
        ascending=False
    )
)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.to_csv(
    '../data/processed/uae_traffic_intelligence_master.csv',
    index=False)



In [ ]:
dubai = df[df['City'] == 'dubai']

dubai_hourly = (
    dubai.groupby('Hour')[
        [
            'TrafficFrictionScore',
            'TravelTimeInflationPct',
            'JamLengthKm',
            'TravelDelayMinutes'
        ]
    ]
    .mean()
    .round(2)
)

print(dubai_hourly)


In [ ]:
all_emirates = (
    df.groupby(['City', 'Hour'])[
        [
            'TrafficFrictionScore',
            'TravelTimeInflationPct',
            'JamLengthKm',
            'TravelDelayMinutes'
        ]
    ]
    .mean()
    .round(2)
)

print(all_emirates)

In [ ]:
for city in df['City'].unique():

    city_df = df[df['City'] == city]

    peak = (
        city_df.groupby('Hour')[
            'TrafficFrictionScore'
        ]
        .mean()
        .max()
    )

    low = (
        city_df.groupby('Hour')[
            'TrafficFrictionScore'
        ]
        .mean()
        .min()
    )

    ratio = round(
        peak / max(low, 0.01),
        2
    )

    print(
        f"{city}: Peak/Low Ratio = {ratio}"
    )

In [ ]:
rush_test = (
    dubai.groupby('Hour')[
        'TrafficFrictionScore'
    ]
    .mean()
    .sort_values(ascending=False)
)

print(rush_test)

In [ ]:
import matplotlib.pyplot as plt

dubai.groupby('Hour')[
    'TrafficFrictionScore'
].mean().plot(
    figsize=(12,5),
    marker='o'
)

plt.title(
    "Dubai Hourly Traffic Friction"
)

plt.xlabel("Hour")

plt.ylabel(
    "Traffic Friction"
)

plt.grid(True)

plt.show()

In [ ]:
target_h = 15
selected_city = "dubai"

In [ ]:
future_hours = [
    target_h,
    (target_h + 1) % 24,
    (target_h + 2) % 24,
    (target_h + 3) % 24
]

print(future_hours)

In [ ]:
future_df = df[
    (df['City'] == selected_city)
    &
    (df['Hour'].isin(future_hours))
]

print(
    future_df[
        [
            'Hour',
            'TrafficFrictionScore',
            'TravelDelayMinutes',
            'JamLengthKm'
        ]
    ]
    .head(20)
)

In [ ]:
future_projection = (

    future_df

    .groupby('Hour')[
        [
            'TrafficFrictionScore',
            'TravelDelayMinutes',
            'JamLengthKm'
        ]
    ]

    .mean()

    .round(2)

    .reset_index()
)

print(future_projection)

In [ ]:
future_projection['ForecastSeverity'] = (

    future_projection['TrafficFrictionScore']

    +

    (
        future_projection['TravelDelayMinutes']
        * 4
    )

    +

    (
        future_projection['JamLengthKm']
        / 120
    )

).round(2)

print(future_projection)

In [ ]:
import plotly.express as px

fig = px.line(

    future_projection,

    x='Hour',

    y='ForecastSeverity',

    markers=True,

    title='Mobility Forecast Trajectory'

)

fig.update_layout(

    template='plotly_dark',

    xaxis_title='Future Hour',

    yaxis_title='Forecast Severity',

    title_x=0.5,

    height=500

)

fig.show()

In [ ]:
print(df.columns)

In [ ]:
def get_network_diagnosis(

    friction,
    delay,
    queue

):

    # Severe Congestion

    if (
        friction > 10
        and delay > 2
        and queue > 450
    ):

        return (
            "Severe Congestion Pressure",
            "Network operating near peak saturation levels."
        )


    # Widespread Congestion

    elif (
        queue > 300
        and delay > 1
    ):

        return (
            "Widespread Congestion",
            "Broad slowdown spreading across major corridors."
        )


    # Heavy Traffic

    elif (
        friction > 4
        and delay > 0.8
    ):

        return (
            "Heavy Traffic Conditions",
            "Sustained commuter pressure impacting mobility."
        )


    # Moderate Build-Up

    elif (
        friction > 1.5
        or queue > 120
    ):

        return (
            "Moderate Traffic Build-Up",
            "Traffic volume increasing across key routes."
        )


    # Smooth Flow

    else:

        return (
            "Smooth Traffic Flow",
            "Minimal operational disruption."
        )

In [ ]:
typology_df = future_projection.copy()

typology_df['NormFriction'] = (
    typology_df['TrafficFrictionScore']
    /
    typology_df['TrafficFrictionScore'].max()
).round(2)

typology_df['NormDelay'] = (
    typology_df['TravelDelayMinutes']
    /
    typology_df['TravelDelayMinutes'].max()
).round(2)

typology_df['NormQueue'] = (
    typology_df['JamLengthKm']
    /
    typology_df['JamLengthKm'].max()
).round(2)

typology_df['NormRisk'] = (
    typology_df['ForecastSeverity']
    /
    typology_df['ForecastSeverity'].max()
).round(2)

print(typology_df)

In [ ]:
for _, row in typology_df.iterrows():

    diagnosis, explanation = get_network_diagnosis(

        row['TrafficFrictionScore'],
        row['TravelDelayMinutes'],
        row['JamLengthKm']

    )

    print(
        f"Hour {row['Hour']} → {diagnosis}"
    )

    print(
        explanation
    )

    print("------")

In [ ]:
pip install requests

In [ ]:
import requests

In [ ]:
CITY_COORDS = {

    "Dubai": (25.2048, 55.2708),
    "Abu Dhabi": (24.4539, 54.3773),
    "Sharjah": (25.3463, 55.4209),
    "Al Ain": (24.2075, 55.7447),
    "Fujairah": (25.1288, 56.3265),
    "Ras Al Khaimah": (25.8007, 55.9762)

}

In [ ]:
city = "Dubai"

lat, lon = CITY_COORDS[city]

print(lat, lon)

In [ ]:
url = (

    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={lat}&longitude={lon}"
    f"&current="
    f"temperature_2m,"
    f"relative_humidity_2m,"
    f"precipitation,"
    f"cloud_cover,"
    f"wind_speed_10m,"
    f"weather_code"

)

print(url)

In [ ]:
response = requests.get(url)

weather_data = response.json()

print(weather_data)

In [ ]:
current = weather_data['current']

temperature = current['temperature_2m']
humidity = current['relative_humidity_2m']
precipitation = current['precipitation']
cloud_cover = current['cloud_cover']
wind_speed = current['wind_speed_10m']
weather_code = current['weather_code']

print("Temperature:", temperature)
print("Humidity:", humidity)
print("Precipitation:", precipitation)
print("Cloud Cover:", cloud_cover)
print("Wind Speed:", wind_speed)
print("Weather Code:", weather_code)

In [ ]:
def interpret_weather(weather_code,
                      humidity,
                      precipitation,
                      wind_speed,
                      cloud_cover):

    # Clear conditions
    if weather_code == 0 and precipitation == 0:

        return (
            "Clear environmental conditions supporting stable mobility flow."
        )

    # Fog / visibility concerns
    elif humidity >= 85 and cloud_cover >= 40:

        return (
            "Reduced visibility conditions may slow commuter movement across major corridors."
        )

    # Rain conditions
    elif precipitation > 0:

        return (
            "Wet road conditions may be contributing to slower traffic movement."
        )

    # Wind instability
    elif wind_speed >= 30:

        return (
            "Elevated wind conditions may be creating minor operational instability."
        )

    # General cloudy conditions
    elif cloud_cover >= 60:

        return (
            "Dense cloud coverage contributing to reduced environmental visibility."
        )

    else:

        return (
            "Environmental conditions remain within normal operational thresholds."
        )

In [ ]:
weather_narrative = interpret_weather(

    weather_code,
    humidity,
    precipitation,
    wind_speed,
    cloud_cover

)

print(weather_narrative)

In [ ]:
def build_operational_context(

    traffic_state,
    traffic_description,
    weather_narrative

):

    combined_context = f"""

{traffic_state}

{traffic_description}

Environmental Context:
{weather_narrative}

"""

    return combined_context.strip()

In [ ]:
traffic_state = "Smooth Traffic Flow"

traffic_description = (
    "Calm network operations. Movement remains fluid "
    "across major commuter corridors."
)

combined_output = build_operational_context(

    traffic_state,
    traffic_description,
    weather_narrative

)

print(combined_output)

In [ ]:
API_KEY = "YgnGll8WdUyY28iDda8Hisc0bkXDsGr8"

url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?key={API_KEY}&point=25.2048,55.2708"

response = requests.get(url)

traffic_data = response.json()

traffic_data

In [ ]:
live_flow = traffic_data['flowSegmentData']

current_speed = live_flow['currentSpeed']
free_flow_speed = live_flow['freeFlowSpeed']

current_time = live_flow['currentTravelTime']
free_flow_time = live_flow['freeFlowTravelTime']

road_closed = live_flow['roadClosure']

congestion_pct = round(
    (
        1 - (current_speed / free_flow_speed)
    ) * 100,
    1
)

delay_seconds = current_time - free_flow_time

print("LIVE TRAFFIC SUMMARY")
print("----------------------")

print(f"Current Speed: {current_speed} km/h")
print(f"Free Flow Speed: {free_flow_speed} km/h")

print(f"Congestion Level: {congestion_pct}%")
print(f"Travel Delay: {delay_seconds} sec")

print(f"Road Closed: {road_closed}")

In [ ]:
dubai_zones = {

    "Downtown Dubai": (25.2048, 55.2708),

    "Dubai Marina": (25.0800, 55.1400),

    "Business Bay": (25.1867, 55.2644),

    "Deira": (25.2711, 55.3075),

    "Jumeirah": (25.2040, 55.2477),

    "JLT": (25.0692, 55.1413),

    "Dubai Airport": (25.2532, 55.3657),

    "Sheikh Zayed Road": (25.2176, 55.2797)

}

In [ ]:
import requests
import pandas as pd

API_KEY = "YgnGll8WdUyY28iDda8Hisc0bkXDsGr8"

live_results = []

for zone, coords in dubai_zones.items():

    lat, lon = coords

    url = (
        f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
        f"?key={API_KEY}&point={lat},{lon}"
    )

    response = requests.get(url)

    data = response.json()

    flow = data['flowSegmentData']

    current_speed = flow['currentSpeed']

    free_flow_speed = flow['freeFlowSpeed']

    current_tt = flow['currentTravelTime']

    free_tt = flow['freeFlowTravelTime']

    congestion_pct = round(
        (
            1 - (current_speed / free_flow_speed)
        ) * 100,
        1
    )

    delay_sec = current_tt - free_tt

    live_results.append({

        "Zone": zone,

        "CurrentSpeed": current_speed,

        "FreeFlowSpeed": free_flow_speed,

        "CongestionPct": congestion_pct,

        "DelaySeconds": delay_sec,

        "RoadClosed": flow['roadClosure']

    })

live_df = pd.DataFrame(live_results)

live_df

In [ ]:
def classify_live_traffic(congestion, delay, road_closed):

    if road_closed:
        return "Critical Road Disruption"

    elif congestion >= 45:
        return "Severe Congestion Pressure"

    elif congestion >= 25:
        return "Widespread Congestion"

    elif congestion >= 10:
        return "Moderate Traffic Build-Up"

    else:
        return "Smooth Traffic Flow"


live_df['LiveTrafficState'] = live_df.apply(

    lambda row:

    classify_live_traffic(

        row['CongestionPct'],
        row['DelaySeconds'],
        row['RoadClosed']

    ),

    axis=1

)

live_df

In [ ]:
zone_coords = {

    "Downtown Dubai": (25.2048, 55.2708),

    "Dubai Marina": (25.0800, 55.1400),

    "Business Bay": (25.1867, 55.2644),

    "Deira": (25.2711, 55.3075),

    "Jumeirah": (25.2040, 55.2477),

    "JLT": (25.0692, 55.1413),

    "Dubai Airport": (25.2532, 55.3657),

    "Sheikh Zayed Road": (25.2176, 55.2797)

}

live_df['Latitude'] = live_df['Zone'].apply(
    lambda x: zone_coords[x][0]
)

live_df['Longitude'] = live_df['Zone'].apply(
    lambda x: zone_coords[x][1]
)

live_df

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(

    live_df,

    lat="Latitude",

    lon="Longitude",

    color="CongestionPct",

    size="CongestionPct",

    hover_name="Zone",

    hover_data={

        "CurrentSpeed": True,

        "FreeFlowSpeed": True,

        "DelaySeconds": True,

        "LiveTrafficState": True,

        "Latitude": False,

        "Longitude": False

    },

    color_continuous_scale="RdYlGn_r",

    zoom=10,

    height=700,

    title="Live Dubai Traffic Intelligence Map"

)

fig.update_layout(

    mapbox_style="carto-darkmatter",

    margin={"r":0,"t":50,"l":0,"b":0}

)

fig.show()

In [ ]:
flow = traffic_data["flowSegmentData"]

flow

In [ ]:
print("Current Speed:", flow["currentSpeed"])
print("Free Flow Speed:", flow["freeFlowSpeed"])
print("Current Travel Time:", flow["currentTravelTime"])
print("Free Flow Travel Time:", flow["freeFlowTravelTime"])
print("Road Closure:", flow["roadClosure"])

In [ ]:
flow["currentSpeed"]

In [ ]:
flow["freeFlowSpeed"]

In [ ]:
current_speed = flow["currentSpeed"]

free_flow_speed = flow["freeFlowSpeed"]

congestion_pct = (
    (free_flow_speed - current_speed)
    / free_flow_speed
) * 100

print(round(congestion_pct, 1))

In [ ]:
current_tt = flow["currentTravelTime"]

freeflow_tt = flow["freeFlowTravelTime"]

delay_seconds = current_tt - freeflow_tt

delay_minutes = delay_seconds / 60

print("Delay Seconds:", delay_seconds)

print("Delay Minutes:", round(delay_minutes, 1))

In [ ]:
if congestion_pct < 10:
    state = "Smooth Traffic Flow"

elif congestion_pct < 25:
    state = "Moderate Traffic Build-Up"

elif congestion_pct < 45:
    state = "Widespread Congestion"

else:
    state = "Severe Congestion Pressure"

print(state)

In [ ]:
print("========== LIVE KPI VALIDATION ==========")

print("Current Speed:", current_speed, "km/h")

print("Free Flow Speed:", free_flow_speed, "km/h")

print("Congestion:", round(congestion_pct,1), "%")

print("Operational Delay:", round(delay_minutes,1), "min")

print("Operational State:", state)

print("========================================")

In [ ]:
import requests

def validate_tomtom_data(api_key, lat, lon):
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?key={api_key}&point={lat},{lon}"
    
    try:
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            print(f"Error fetching data: {response.status_code} - {response.text}")
            return
            
        data = response.json()['flowSegmentData']
        
        # 1. Extract Raw Telemetry
        c_speed = data['currentSpeed']
        f_speed = data['freeFlowSpeed']
        c_time = data['currentTravelTime']
        f_time = data['freeFlowTravelTime']
        
        # 2. APPLY CORRECTED MATH (Capacity Loss)
        if f_speed > 0:
            congestion_pct = (1 - (c_speed / f_speed)) * 100
        else:
            congestion_pct = 0
            
        # 3. Calculate Delay in Minutes
        delay_sec = c_time - f_time
        delay_min = round(delay_sec / 60, 1)
        
        # 4. Print Executive Output
        print("="*40)
        print("🚦 TOMTOM API VALIDATION SCRIPT")
        print("="*40)
        print(f"🚗 Live Speed          : {c_speed} km/h")
        print(f"   Free flow           : {f_speed} km/h")
        print("-" * 40)
        print(f"📊 Congestion Pressure : {congestion_pct:.1f}%")
        print("   (vs free-flow baseline)")
        print("-" * 40)
        print(f"⏱️ Operational Delay   : {delay_min} min")
        print("   (above free-flow travel time)")
        print("="*40)

    except Exception as e:
        print(f"Connection failed: {e}")

# ==========================================
# TEST CONFIGURATION
# ==========================================
API_KEY = "YgnGll8WdUyY28iDda8Hisc0bkXDsGr8"  # <-- Paste your API Key here
LAT = 25.1972                    # <-- Paste your test Latitude here
LON = 55.2744                    # <-- Paste your test Longitude here

validate_tomtom_data(API_KEY, LAT, LON)